# 05 — Anatomy of `configs/searchqa/default.yaml`

SkillOpt is config-driven: `python scripts/train.py --config <path>.yaml`. Every knob — model, train size, batch size, edit budget, scheduler — lives in YAML.

This notebook loads the SearchQA default config, walks every key, shows the `_base_` inheritance, and composes a custom override **in Python** rather than YAML-editing. No LLM calls.

In [1]:
import os
from pathlib import Path

import yaml

SKILLOPT = Path("/Users/mhuang/Documents/GitHub/SkillOpt")
CONFIG_DIR = SKILLOPT / "configs"

print("config dir:", CONFIG_DIR)
print()
for bench in sorted(os.listdir(CONFIG_DIR)):
    sub = CONFIG_DIR / bench
    if sub.is_dir():
        files = [f for f in os.listdir(sub) if f.endswith(".yaml")]
        print(f"  {bench:<25} {files}")

config dir: /Users/mhuang/Documents/GitHub/SkillOpt/configs

  _base_                    ['default.yaml']
  alfworld                  ['default.yaml']
  docvqa                    ['default.yaml']
  livemathematicianbench    ['default.yaml']
  officeqa                  ['default.yaml']
  searchqa                  ['default.yaml']
  spreadsheetbench          ['default.yaml']


In [2]:
base_cfg = yaml.safe_load((CONFIG_DIR / "_base_" / "default.yaml").read_text())
searchqa_cfg = yaml.safe_load((CONFIG_DIR / "searchqa" / "default.yaml").read_text())

print("=== _base_/default.yaml ===")
print(f"top-level keys: {sorted(base_cfg.keys())}")
print(f"shape: {len(yaml.dump(base_cfg))} chars of YAML")
print()
print("=== searchqa/default.yaml (overrides + new fields) ===")
print(f"top-level keys: {sorted(searchqa_cfg.keys())}")
print(f"_base_ pointer: {searchqa_cfg.get('_base_')!r}")

=== _base_/default.yaml ===
top-level keys: ['env', 'evaluation', 'gradient', 'model', 'optimizer', 'train']
shape: 2291 chars of YAML

=== searchqa/default.yaml (overrides + new fields) ===
top-level keys: ['_base_', 'env', 'evaluation', 'gradient', 'model', 'optimizer', 'train']
_base_ pointer: '../_base_/default.yaml'


In [3]:
# Resolve _base_ manually to show what the trainer would see
def resolve_base(cfg_path: Path) -> dict:
    cfg = yaml.safe_load(cfg_path.read_text())
    base_ref = cfg.pop("_base_", None)
    if base_ref:
        parent = (cfg_path.parent / base_ref).resolve()
        merged = resolve_base(parent)
    else:
        merged = {}
    # Deep merge
    for k, v in cfg.items():
        if k in merged and isinstance(v, dict) and isinstance(merged[k], dict):
            merged[k] = {**merged[k], **v}
        else:
            merged[k] = v
    return merged


resolved = resolve_base(CONFIG_DIR / "searchqa" / "default.yaml")
print("resolved searchqa config — top sections + key counts:")
for k, v in resolved.items():
    if isinstance(v, dict):
        print(f"  {k:<15} dict with {len(v)} keys: {list(v.keys())}")
    else:
        print(f"  {k:<15} {type(v).__name__:<8} {repr(v)[:60]}")

resolved searchqa config — top sections + key counts:
  model           dict with 41 keys: ['backend', 'optimizer', 'target', 'optimizer_backend', 'target_backend', 'reasoning_effort
', 'rewrite_reasoning_effort', 'rewrite_max_completion_tokens', 'codex_exec_path', 'codex_exec_sandbox', 'codex_exec_profile', '
codex_exec_full_auto', 'codex_exec_reasoning_effort', 'codex_exec_use_sdk', 'codex_exec_network_access', 'codex_exec_web_search'
, 'codex_exec_approval_policy', 'claude_code_exec_path', 'claude_code_exec_profile', 'claude_code_exec_use_sdk', 'claude_code_ex
ec_effort', 'claude_code_exec_max_thinking_tokens', 'codex_trace_to_optimizer', 'azure_openai_endpoint', 'azure_openai_api_versi
on', 'azure_openai_api_key', 'azure_openai_auth_mode', 'azure_openai_ad_scope', 'azure_openai_managed_identity_client_id', 'opti
mizer_azure_openai_endpoint', 'optimizer_azure_openai_api_version', 'optimizer_azure_openai_api_key', 'optimizer_azure_openai_au
th_mode', 'optimizer_azure_openai_ad_scope'

## Drilling into each section

In [4]:
for section in ("model", "train", "gradient", "optimizer", "evaluation", "env"):
    sub = resolved.get(section, {})
    print(f"--- {section} ---")
    print(yaml.dump(sub, default_flow_style=False, sort_keys=True).rstrip())
    print()

--- model ---
azure_openai_ad_scope: https://cognitiveservices.azure.com/.default
azure_openai_api_key: ''
azure_openai_api_version: 2024-12-01-preview
azure_openai_auth_mode: azure_cli
azure_openai_endpoint: ''
azure_openai_managed_identity_client_id: ''
backend: azure_openai
claude_code_exec_effort: medium
claude_code_exec_max_thinking_tokens: 16384
claude_code_exec_path: claude
claude_code_exec_profile: ''
claude_code_exec_use_sdk: auto
codex_exec_approval_policy: never
codex_exec_full_auto: false
codex_exec_network_access: false
codex_exec_path: codex
codex_exec_profile: ''
codex_exec_reasoning_effort: none
codex_exec_sandbox: workspace-write
codex_exec_use_sdk: auto
codex_exec_web_search: false
codex_trace_to_optimizer: true
optimizer: gpt-5.5
optimizer_azure_openai_ad_scope: https://cognitiveservices.azure.com/.default
optimizer_azure_openai_api_key: ''
optimizer_azure_openai_api_version: 2024-12-01-preview
optimizer_azure_openai_auth_mode: azure_cli
optimizer_azure_openai_endpoi

## Compose a custom config in Python — for a tiny smoke run

The point: you don't have to edit YAML by hand. Override exactly the fields you care about, in-kernel, and pass the result to the Trainer.

In [5]:
from copy import deepcopy

cfg = deepcopy(resolved)
# Aim for a 1-step smoke run
cfg["model"]["model_backend"] = "claude_chat"
cfg["train"]["train_size"] = 4
cfg["train"]["batch_size"] = 2
cfg["train"]["accumulation"] = 1
cfg["train"]["epochs"] = 1
cfg["gradient"]["minibatch_size"] = 2
cfg["gradient"]["merge_batch_size"] = 2
cfg["optimizer"]["learning_rate"] = 2
cfg["evaluation"]["sel_env_num"] = 2
cfg["evaluation"]["test_env_num"] = 0
cfg["env"]["workers"] = 1
cfg["env"]["limit"] = 4

print("=== smoke config (the overrides) ===")
for section in ("model", "train", "gradient", "optimizer", "evaluation", "env"):
    print(f"[{section}]")
    print(yaml.dump(cfg[section], default_flow_style=False).rstrip())
    print()

=== smoke config (the overrides) ===
[model]
azure_openai_ad_scope: https://cognitiveservices.azure.com/.default
azure_openai_api_key: ''
azure_openai_api_version: 2024-12-01-preview
azure_openai_auth_mode: azure_cli
azure_openai_endpoint: ''
azure_openai_managed_identity_client_id: ''
backend: azure_openai
claude_code_exec_effort: medium
claude_code_exec_max_thinking_tokens: 16384
claude_code_exec_path: claude
claude_code_exec_profile: ''
claude_code_exec_use_sdk: auto
codex_exec_approval_policy: never
codex_exec_full_auto: false
codex_exec_network_access: false
codex_exec_path: codex
codex_exec_profile: ''
codex_exec_reasoning_effort: none
codex_exec_sandbox: workspace-write
codex_exec_use_sdk: auto
codex_exec_web_search: false
codex_trace_to_optimizer: true
model_backend: claude_chat
optimizer: gpt-5.5
optimizer_azure_openai_ad_scope: https://cognitiveservices.azure.com/.default
optimizer_azure_openai_api_key: ''
optimizer_azure_openai_api_version: 2024-12-01-preview
optimizer_azure

In [6]:
# Save the composed config to a file so scripts/train.py can also consume it
smoke_path = Path("/tmp/abook_searchqa_smoke.yaml")
smoke_path.write_text(yaml.dump(cfg, default_flow_style=False, sort_keys=True))
print(f"wrote: {smoke_path}  ({smoke_path.stat().st_size} bytes)")
print()
print("Now `python scripts/train.py --config /tmp/abook_searchqa_smoke.yaml ...` would consume this.")
print("Notebook 06 actually drives it.")

wrote: /tmp/abook_searchqa_smoke.yaml  (2435 bytes)

Now `python scripts/train.py --config /tmp/abook_searchqa_smoke.yaml ...` would consume this.
Notebook 06 actually drives it.


## Recap — what this notebook proved

The path this notebook walked, in the order the cells walked it:

- 05 — Anatomy of `configs/searchqa/default.yaml`

Each step above was a real cell above. Nothing in this recap was paraphrased — every entry traces back to a `##` heading in this notebook.


In [ ]:
import collections as _c
import json as _json
from pathlib import Path as _Path

_nb_path = _Path("/Users/mhuang/Documents/GitHub/abook/notebooks/skillopt/05-config-yaml.ipynb")
_nb = _json.loads(_nb_path.read_text())
_cells = _nb["cells"]

# Cell type breakdown
_type_counts = _c.Counter(c["cell_type"] for c in _cells)

# Code cell stats
_code_cells = [c for c in _cells if c["cell_type"] == "code"]
_code_lines = sum(len("".join(c["source"]).splitlines()) for c in _code_cells)
_md_chars = sum(len("".join(c["source"])) for c in _cells if c["cell_type"] == "markdown")

# Output mime types seen
_mimes = _c.Counter()
_executed = 0
_errored = 0
for c in _code_cells:
    if c.get("execution_count") is not None:
        _executed += 1
    for out in c.get("outputs", []) or []:
        if out.get("output_type") == "error":
            _errored += 1
        for k in (out.get("data") or {}).keys():
            _mimes[k] += 1
        if out.get("output_type") == "stream":
            _mimes[f"stream:{out.get('name', 'stdout')}"] += 1

print(f"notebook        : {_nb_path.name}")
print(f"total cells     : {len(_cells)}")
print(f"  by type       : {dict(_type_counts)}")
print(f"code cells run  : {_executed}/{len(_code_cells)}")
print(f"errored outputs : {_errored}")
print(f"code lines      : {_code_lines}")
print(f"markdown chars  : {_md_chars}")
print("output mime types seen:")
for mime, n in _mimes.most_common():
    print(f"  {n:>3}  {mime}")

## Data sources

| Source | Path |
|---|---|
| Base config | `~/Documents/GitHub/SkillOpt/configs/_base_/default.yaml` |
| Benchmark override | `~/Documents/GitHub/SkillOpt/configs/searchqa/default.yaml` |
| Composed smoke config | `/tmp/abook_searchqa_smoke.yaml` (written by this notebook) |
| `_base_` resolver | local Python function `resolve_base` |

→ **Next:** [`06-quickstart-and-train.ipynb`](06-quickstart-and-train.ipynb) — live training using this smoke config.